In [1]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2
import os

os.makedirs('../data/clean', exist_ok=True)

> Importing required libraries and creating a clean folder to save the clean  csv file

In [3]:
df = pd.read_csv(r"C:\dishu\Cleartrace\data\processed\delhi_aqi_90d.csv")

df['timestamp_hour'] = pd.to_datetime(df['timestamp_hour'])
df = df.sort_values(['station_name', 'timestamp_hour']).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Dtypes of columns to fix:")
print(df[['wind_direction_10m', 'relative_humidity_2m']].dtypes)

Shape: (82004, 44)
Dtypes of columns to fix:
wind_direction_10m      int64
relative_humidity_2m    int64
dtype: object


## Task 1 — Datatype Conversion

wind_direction_10m and relative_humidity_2m are stored as int64.
Converting to float64 for two reasons:
- Consistency with other meteorological columns.
- wind_direction_10m needs float64 for sin/cos calculations.

In [ ]:
df['wind_direction_10m'] = df['wind_direction_10m'].astype(float)
df['relative_humidity_2m'] = df['relative_humidity_2m'].astype(float)

print(df[['wind_direction_10m', 'relative_humidity_2m']].dtypes)

## Task 2 - To add reliability column:
- If the valid AQI hours> 50 % of total hours,  we flag the station as reliable
- If not, we dont

In [5]:
reliability = df.groupby('station_name')['aqi_calculation_valid'].mean()

df['station_reliability'] = df['station_name'].map(reliability) > 0.5

print("Reliability per station:")
print(reliability.sort_values())

Reliability per station:
station_name
Chandni Chowk, Delhi - IITM                         0.077386
IHBAS, Dilshad Garden,New Delhi - CPCB              0.382298
NSIT Dwarka, Delhi - CPCB                           0.445783
New Moti Bagh, Delhi - MHUA                         0.539388
Alipur, Delhi - DPCC                                0.700185
Sonia Vihar, Delhi - DPCC                           0.715941
Anand Vihar, New Delhi - DPCC                       0.721501
Mandir Marg, New Delhi - DPCC                       0.741891
North Campus, DU, Delhi - IMD                       0.745134
Vivek Vihar, Delhi - DPCC                           0.745598
Burari Crossing, New Delhi - IMD                    0.753012
Dr. Karni Singh Shooting Range, Delhi - DPCC        0.753939
Najafgarh, Delhi - DPCC                             0.764597
Jahangirpuri, Delhi - DPCC                          0.765060
ITO, New Delhi - CPCB                               0.765524
Punjabi Bagh, Delhi - DPCC                     

## Findings:
> We conclude that the 3 stations-
- Chandni Chowk, Delhi - IITM
- IHBAS, Dilshad Garden,New Delhi - CPCB
- NSIT Dwarka, Delhi - CPCB
> are  marked as unreliable, which confirms our findings from missingno heatmap

## Task 3 — Missing Value Imputation

Strategy based on gap size:
- Gaps ≤ 6 hours → linear interpolation per station (limit=6)
- Gaps > 6 hours → proxy fill from nearest reliable station

Linear interpolation draws a straight line between last known 
and next known value — valid for short gaps since pollution 
changes gradually. Long gaps need proxy fill from nearby stations.

Chandni Chowk's 2-month gaps remain NaN intentionally — 
station is already flagged as low reliability.

In [7]:
pollutants = ['pm25', 'pm10', 'no2', 'co', 'so2', 'o3']

df_clean = df.copy()

for col in pollutants:
    df_clean[col] = (
        df_clean.groupby('station_name')[col]
        .transform(lambda x: x.interpolate(method='linear', limit=6))
    )

print("Missing values after interpolation:")
print(df_clean[pollutants].isnull().sum())
print(f"\nBefore: {df[pollutants].isnull().sum().sum()}")
print(f"After:  {df_clean[pollutants].isnull().sum().sum()}")

Missing values after interpolation:
pm25     6408
pm10     7842
no2      6447
co       6588
so2     20181
o3       7859
dtype: int64

Before: 129998
After:  55325


In [8]:
remaining = df_clean[pollutants].isnull().sum()
print("Still needs proxy fill (gaps > 6 hours):")
print(remaining)
print(f"\nTotal remaining: {remaining.sum()}")

Still needs proxy fill (gaps > 6 hours):
pm25     6408
pm10     7842
no2      6447
co       6588
so2     20181
o3       7859
dtype: int64

Total remaining: 55325
